# Projet : Prédiction du niveau de pauvreté des ménages marocains

Ce notebook applique diverses techniques d'apprentissage non-supervisé (Clustering) pour analyser le niveau de pauvreté des ménages marocains, en se basant sur le fichier de données `indic-soc-niveau-vie-equipements-base-mef2014.xls`.

**Étapes clés :**
1. Installation et importation des bibliothèques.
2. Chargement et exploration du dataset.
3. Prétraitement des données.
4. Réduction de dimension avec PCA pour la visualisation.
5. Clustéring des données (K-Means, DBSCAN, HAC, GMM).
6. Évaluation et Visualisation des clusters.

In [ ]:
# 1. Installation des dépendances (utile pour Google Colab)
!pip install pandas numpy matplotlib seaborn scikit-learn xlrd openpyxl --quiet
print('✅ Dépendances installées')

In [ ]:
# 2. Importation des bibliothèques nécessaires
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

print('✅ Imports OK')

---
## 1. Chargement et Exploration du Dataset

In [ ]:
# Charger les données
file_path = 'indic-soc-niveau-vie-equipements-base-mef2014.xls'
try:
    data = pd.read_excel(file_path)
    print(f'✅ Dataset chargé avec succès : {data.shape[0]} lignes et {data.shape[1]} colonnes.')
except Exception as e:
    print('❌ Erreur lors du chargement, veuillez vérifier le chemin du fichier XLS.')
    print(e)

# Afficher les premières lignes
data.head()

In [ ]:
# Exploration globale
data.info()
print('\nRésumé statistique :')
display(data.describe())

---
## 2. Prétraitement des Données
Le nettoyage dépend de la nature des données. Ici nous : 
- Gardons uniquement les variables numériques. 
- Imputons les valeurs manquantes par la médiane. 
- Normalisons avec **StandardScaler**.

In [ ]:
def preprocess_data(df):
    df_num = df.select_dtypes(include=[np.number]).copy()
    
    col_with_na = df_num.columns[df_num.isnull().any()].tolist()
    if len(col_with_na) > 0:
        for col in col_with_na:
            df_num[col].fillna(df_num[col].median(), inplace=True)
            
    sc = StandardScaler()
    X_scaled = sc.fit_transform(df_num)
    return X_scaled, df_num

X, data_num = preprocess_data(data)
print('✅ Données nettoyées et normalisées. Shape :', X.shape)

---
## 3. Réduction de Dimension (PCA) pour la Visualisation

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X)

var_expl = pca2.explained_variance_ratio_ * 100
print(f'✅ PCA appliqué. Variance : {var_expl[0]:.2f}% et {var_expl[1]:.2f}%')

plt.figure(figsize=(7, 5))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], alpha=0.6, s=30, color='gray')
plt.title('Projection 2D des ménages (PCA)')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.show()

---
## 4. Fonctions Utilitaires d'Évaluation et de Visualisation

In [ ]:
def eval_clustering(X, labels, name='Modèle'):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    mask = labels != -1
    if n_clusters >= 2 and mask.sum() > 1:
        sil = silhouette_score(X[mask], labels[mask])
        print(f'{name:20s} | Clusters={n_clusters} | Silhouette={sil:.4f}')
        return sil
    return None

def plot_clusters(X_2d, labels, title):
    plt.figure(figsize=(8, 5))
    unique = np.unique(labels)
    cmap = cm.get_cmap('tab10', len(unique))
    for i, lbl in enumerate(unique):
        mask = labels == lbl
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], s=30, 
                    color='black' if lbl == -1 else cmap(i), 
                    label=f'Cluster {lbl}' if lbl >= 0 else 'Bruit', alpha=0.7)
    plt.title(title)
    plt.legend()
    plt.show()

---
## 5. Algorithmes de Clustering
### 5.1 K-Means

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)

plt.plot(list(K_range), inertias, marker='o', lw=2)
plt.title('Méthode du Coude')
plt.show()

In [ ]:
k_opt = 3 # A ajuster selon le coude
km = KMeans(n_clusters=k_opt, init='k-means++', n_init=10, random_state=42)
labels_km = km.fit_predict(X)
sil_km = eval_clustering(X, labels_km, 'K-Means')
plot_clusters(X_pca2, labels_km, 'K-Means')

### 5.2 DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=2.0, min_samples=5)
labels_db = db.fit_predict(X)
sil_db = eval_clustering(X, labels_db, 'DBSCAN')
plot_clusters(X_pca2, labels_db, 'DBSCAN')

### 5.3 Agglomerative Clustering (HAC)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
hac = AgglomerativeClustering(n_clusters=k_opt, linkage='ward')
labels_hac = hac.fit_predict(X)
sil_hac = eval_clustering(X, labels_hac, 'HAC')
plot_clusters(X_pca2, labels_hac, 'HAC (Ward)')

### 5.4 Gaussian Mixture Model (GMM)

In [ ]:
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=k_opt, random_state=42)
labels_gmm = gmm.fit_predict(X)
sil_gmm = eval_clustering(X, labels_gmm, 'GMM')
plot_clusters(X_pca2, labels_gmm, 'GMM')

---
## 6. Synthèse et Comparaison

In [ ]:
algos = ['K-Means', 'DBSCAN', 'HAC', 'GMM']
scores = [sil_km or 0, sil_db or 0, sil_hac or 0, sil_gmm or 0]
plt.barh(algos, scores, color='skyblue')
plt.title('Comparaison Silhouette Scores')
plt.show()

---
## Profilage
Rattachement des labels K-Means aux données pour interpréter les profils de pauvreté.

In [ ]:
np.warnings.filterwarnings('ignore')
data['Cluster_ID'] = labels_km
print('\nRépartition des ménages par cluster :')
print(data['Cluster_ID'].value_counts())

print('\nProfil moyen des clusters :')
display(data.groupby('Cluster_ID').mean().round(2))